In [0]:
%run ../../00_common/data_utils

In [0]:
def update_master_consumer_table(anonymization_df, task_id):
    """
    更新 t_master_consumer：
      - consumermdmkey = New_UniversalKey
      - scon_srcc_action = 'DELETE'
      - 按 5.1_generate_master_consumer_tables.py 的 DELETE 逻辑清空 PII 字段
      - task_id = 当前 task_id
    按 BusinessKey (market, brand, sourcesystemcode, consumerid) 匹配。
    """
    golden_db = get_env_config('golden_consumer_master_database')
    master_consumer_table = f"{golden_db}.t_master_consumer"

    source_df = anonymization_df.select(
        F.col("MarketCode"),
        F.col("BrandCode"),
        F.col("SourceSystemCode"),
        F.col("ConsumerId"),
        F.col("New_UniversalKey")
    ).distinct()

    if source_df.isEmpty():
        print("No matched anonymization records, skip master consumer update")
        return

    master_consumer_delta = DeltaTable.forName(spark, master_consumer_table)
    (
        master_consumer_delta.alias("target")
        .merge(
            source_df.alias("source"),
            """
            target.scon_srcs_code = source.SourceSystemCode AND
            target.scon_mrkt_code = source.MarketCode AND
            target.scon_brnd_code = source.BrandCode AND
            target.scon_consumerid = source.ConsumerId
            """
        )
        .whenMatchedUpdate(set={
            "consumermdmkey": F.col("source.New_UniversalKey"),
            "scon_srcc_action": F.lit("DELETE"),
            "scon_salutation": F.lit(""),
            "scon_englishfirstname": F.lit(""),
            "scon_englishmiddlename": F.lit(""),
            "scon_englishlastname": F.lit(""),
            "scon_englishfullname": F.lit(""),
            "scon_localfirstname": F.lit(""),
            "scon_localmiddlename": F.lit(""),
            "scon_locallastname": F.lit(""),
            "scon_localfullname": F.lit(""),
            "scon_localfirstname2": F.lit(""),
            "scon_localmiddlename2": F.lit(""),
            "scon_locallastname2": F.lit(""),
            "scon_localfullname2": F.lit(""),
            "scon_identitynum": F.lit(""),
            "scon_passportnum": F.lit(""),
            "scon_socialsecuritynum": F.lit(""),
            "scon_birthday": F.lit(None),
            "task_id": F.lit(task_id),
            "scon_update_dt": F.current_timestamp(),
            "scon_update_uid": F.lit("ELC")
        })
        .execute()
    )

In [0]:
def _build_pii_update_df(master_df, base_df, id_col, mrkt_col, fk_col, filter_expr=None):
    """
    构造单个 PII 表的待更新数据集：按 (id, market) 聚合最大 delete_timestamp。
    通过 fk_col（如 scme_scon_id）关联 master_consumer 的 scon_id。
    """
    df = (
        master_df.alias("m")
        .join(
            base_df.alias("base"),
            (F.col(f"m.{fk_col}") == F.col("base.scon_id")) &
            (F.col(f"m.{mrkt_col}") == F.col("base.scon_mrkt_code")),
            "inner"
        )
    )
    if filter_expr is not None:
        df = df.where(filter_expr)
    return (
        df.groupBy(f"m.{id_col}", f"m.{mrkt_col}")
        .agg(F.max("base.delete_timestamp").alias("delete_timestamp"))
        .cache()
    )


def _clear_pii_table(master_df, base_df, table_name, id_col, mrkt_col, fk_col, filter_expr, update_set):
    """
    对单个 PII 表执行 Delta merge update，返回更新行数。
    """
    to_update = _build_pii_update_df(master_df, base_df, id_col, mrkt_col, fk_col, filter_expr)

    update_count = to_update.count()
    if update_count > 0:
        (
            DeltaTable.forName(spark, table_name).alias("target")
            .merge(
                to_update.alias("source"),
                f"target.{id_col} = source.{id_col} AND target.{mrkt_col} = source.{mrkt_col}"
            )
            .whenMatchedUpdate(set=update_set)
            .execute()
        )

    to_update.unpersist()
    return update_count


def update_master_other_tables(anonymization_df):
    """
    清空 t_master_phone / t_master_emedia / t_master_address / t_master_optin 中的 PII。
    参考 05_Master_Data_Generate/5.1_generate_master_consumer_tables.py 的 DELETE 逻辑，
    使用 current_timestamp() 作为 source timestamp。
    """
    golden_db = get_env_config('golden_consumer_master_database')
    master_consumer_table = f"{golden_db}.t_master_consumer"
    master_emedia_table = f"{golden_db}.t_master_emedia"
    master_phone_table = f"{golden_db}.t_master_phone"
    master_address_table = f"{golden_db}.t_master_address"
    master_optin_table = f"{golden_db}.t_master_optin"

    master_consumer_df = spark.table(master_consumer_table)
    master_emedia_df = spark.table(master_emedia_table)
    master_phone_df = spark.table(master_phone_table)
    master_address_df = spark.table(master_address_table)
    master_optin_df = spark.table(master_optin_table)

    # 构造 DELETE 基础数据集（scon_id + market + 当前时间戳）
    delete_consumer_base_df = (
        master_consumer_df.alias("scon")
        .join(
            anonymization_df.alias("src"),
            (F.col("scon.scon_srcs_code") == F.col("src.SourceSystemCode")) &
            (F.col("scon.scon_mrkt_code") == F.col("src.MarketCode")) &
            (F.col("scon.scon_brnd_code") == F.col("src.BrandCode")) &
            (F.col("scon.scon_consumerid") == F.col("src.ConsumerId")),
            "inner"
        )
        .select(
            F.col("scon.scon_id"),
            F.col("scon.scon_mrkt_code"),
            F.col("scon.SCON_SOURCETIMESTAMP").alias("delete_timestamp")
        )
        .cache()
    )

    if delete_consumer_base_df.isEmpty():
        print("clear-PII completed: emedia=0, phone=0, address=0, optin=0")
        delete_consumer_base_df.unpersist()
        return

    # 1) emedia: 清空 scme_address
    emedia_update_count = _clear_pii_table(
        master_emedia_df, delete_consumer_base_df, master_emedia_table,
        "scme_id", "scme_mrkt_code", "scme_scon_id",
        F.coalesce(F.col("m.scme_address"), F.lit("")) != "",
        {
            "scme_address": F.lit(""),
            "scme_sourcetimestamp": F.col("source.delete_timestamp"),
            "scme_update_dt": F.current_timestamp(),
            "scme_update_uid": F.lit("ELC")
        }
    )

    # 2) phone: 清空 scph_phonenumber
    phone_update_count = _clear_pii_table(
        master_phone_df, delete_consumer_base_df, master_phone_table,
        "scph_id", "scph_mrkt_code", "scph_scon_id",
        F.coalesce(F.col("m.scph_phonenumber"), F.lit("")) != "",
        {
            "scph_phonenumber": F.lit(""),
            "scph_sourcetimestamp": F.col("source.delete_timestamp"),
            "scph_update_dt": F.current_timestamp(),
            "scph_update_uid": F.lit("ELC")
        }
    )

    # 3) address: 清空 scad_address1/2/3
    address_update_count = _clear_pii_table(
        master_address_df, delete_consumer_base_df, master_address_table,
        "scad_id", "scad_mrkt_code", "scad_scon_id",
        F.concat_ws("", F.col("m.scad_address1"), F.col("m.scad_address2"), F.col("m.scad_address3")) != "",
        {
            "scad_address1": F.lit(""),
            "scad_address2": F.lit(""),
            "scad_address3": F.lit(""),
            "scad_sourcetimestamp": F.col("source.delete_timestamp"),
            "scad_update_dt": F.current_timestamp(),
            "scad_update_uid": F.lit("ELC")
        }
    )

    # 4) optin: 将 scop_optin_flag 置 0
    optin_update_count = _clear_pii_table(
        master_optin_df, delete_consumer_base_df, master_optin_table,
        "scop_id", "scop_mrkt_code", "scop_scon_id",
        F.col("m.scop_optin_flag").isNotNull() & (F.col("m.scop_optin_flag") != F.lit(False)),
        {
            "scop_optin_flag": F.lit(False),
            "scop_optin_dt": F.col("source.delete_timestamp"),
            "scop_update_dt": F.current_timestamp(),
            "scop_update_uid": F.lit("ELC")
        }
    )

    print(
        f"clear-PII completed: emedia={emedia_update_count}, "
        f"phone={phone_update_count}, address={address_update_count}, "
        f"optin={optin_update_count}"
    )

    delete_consumer_base_df.unpersist()

In [0]:
def anonymize_master_data(anonymization_df, task_id):
    """
    整合 #6.1 + #6.3:
      - 更新 t_master_consumer (key/action/PII/task_id)
      - 清空 t_master_emedia / phone / address / optin 中的 PII
    """
    update_master_consumer_table(anonymization_df, task_id)
    update_master_other_tables(anonymization_df)

In [0]:
def update_other_brands_task_id(active_df, task_id):
    """
    对于 4.1 方式生成新 Ukey 的记录，
    将同一 market + ukey 下的其他 brand 的 task_id 更新为当前 task_id。
    """
    golden_db = get_env_config('golden_consumer_master_database')
    master_consumer_table = f"{golden_db}.t_master_consumer"
    master_consumer_df = spark.table(master_consumer_table)

    other_brands_df = (
        master_consumer_df.alias("mc")
        .join(
            active_df.alias("src"),
            (F.col("mc.scon_mrkt_code") == F.col("src.MarketCode")) &
            (F.col("mc.consumermdmkey") == F.col("src.Ukey")) &
            (F.col("mc.scon_brnd_code") != F.col("src.BrandCode")),
            "inner"
        )
        .select(
            F.col("mc.scon_id").alias("scon_id"),
            F.col("mc.scon_mrkt_code").alias("scon_mrkt_code")
        )
        .distinct()
    )

    if other_brands_df.isEmpty():
        print("No other brands to update task_id")
        return

    master_consumer_delta = DeltaTable.forName(spark, master_consumer_table)
    (
        master_consumer_delta.alias("target")
        .merge(
            other_brands_df.alias("source"),
            """
            target.scon_id = source.scon_id AND
            target.scon_mrkt_code = source.scon_mrkt_code
            """
        ).whenMatchedUpdate(set={
            "task_id": F.lit(task_id),
            "scon_update_dt": F.current_timestamp(),
            "scon_update_uid": F.lit("ELC")
        })
        .execute()
    )

In [0]:
def delete_old_derived_records(active_df):
    """
    对于 4.1 方式生成新 Ukey 的记录，
    删除 t_derived_consumer_l2 / t_derived_consumer_l3 中残留的
    market + old_ukey(consumermdmkey) + brand 数据。
    """
    if active_df.isEmpty():
        print("No active records, skip derived table deletion")
        return

    golden_db = get_env_config('golden_consumer_master_database')
    l2_table = f"{golden_db}.t_derived_consumer_l2"
    l3_table = f"{golden_db}.t_derived_consumer_l3"

    delete_keys_df = (
        active_df
        .select(
            F.col("MarketCode").alias("scon_mrkt_code"),
            F.col("Ukey").alias("consumermdmkey"),
            F.col("BrandCode").alias("scon_brnd_code")
        )
    )

    for table_name in [l2_table, l3_table]:
        (
            DeltaTable.forName(spark, table_name).alias("target")
            .merge(
                delete_keys_df.alias("source"),
                """
                target.scon_mrkt_code = source.scon_mrkt_code AND
                target.consumermdmkey = source.consumermdmkey AND
                target.scon_brnd_code = source.scon_brnd_code
                """
            )
            .whenMatchedDelete()
            .execute()
        )

In [0]:
def process_anonymization_data(task_id):
    """
    步骤 6.x: 从 anonymization log 读取匹配结果 → 重算 has_other_brands →
    更新 master 表 → 清空 PII → 更新其他 brand task_id → 删除残留 derived 数据。
    """
    anonymization_db = get_env_config('silver_mdm_anonymization_database')
    log_table = f"{anonymization_db}.t_mdm_anonymization_log"

    # 从 log 表读取当前 task 的 anonymization 记录
    log_df = spark.table(log_table) \
        .filter(F.col("status") == ANON_STATUS_IN_PROGRESS) \
        .filter(F.col("task_id") == task_id)

    if log_df.isEmpty():
        print("No anonymization records in log for this task_id, nothing to do")
        return

    anonymization_df = (
        log_df
        .select(
            F.col("ConsumerId"),
            F.col("MarketCode"),
            F.col("BrandCode"),
            F.col("SourceSystemCode"),
            F.col("Old_UniversalKey").alias("Ukey"),
            F.col("New_UniversalKey")
        )
        .checkpoint(eager=True)
    )

    anonymization_count = anonymization_df.count()
    print(f'matched anonymization count: {anonymization_count}')

    print("4. update master table & clear pii for matched anonymization consumer")

    # 6.1 更新 master 表并清空 PII
    anonymize_master_data(anonymization_df, task_id)

    print("5. update task_id for other brand keys")

    # 6.2 更新同 market+ukey 下其他 brand 的 task_id
    # ponytail: Ukey != New_UniversalKey 等价于 has_other_brands=True（4.1场景），避免冗余字段
    active_df = (
        anonymization_df
        .filter(F.col("Ukey") != F.col("New_UniversalKey"))
        .select("MarketCode", "BrandCode", "Ukey")
        .distinct()
    )

    active_df = active_df.checkpoint(eager=True)
    active_count = active_df.count()
    print(f'keys count: {active_count}')

    update_other_brands_task_id(active_df, task_id)

    print("6. delete old derived data for anonymized keys")
    print(f'keys count: {active_count}')

    # 6.3 删除 derived l2/l3 中残留的老 ukey + brand 数据
    delete_old_derived_records(active_df)

In [0]:
task_id = dbutils.widgets.get("task_id")
print(f"task_id: {task_id}")

step_name = "process_anonymization_data"
step_num = "02"
project = "dataanonymization"
log_table_name = f"{get_env_config('config_database')}.t_task_step_log"

start_time = datetime.now()
status = "SUCCESS"
message = "completed"

try:
    spark.sparkContext.setCheckpointDir(f"{get_env_config('checkpoint_path_consumer_master')}/{task_id}")
    process_anonymization_data(task_id)
except Exception as e:
    status = "FAILED"
    message = f"{type(e).__name__}: {str(e)}"
    raise
finally:
    end_time = datetime.now()
    append_step_log(
        log_table_name=log_table_name,
        task_id=task_id,
        step_num=step_num,
        step_name=step_name,
        start_time=start_time,
        end_time=end_time,
        status=status,
        message=message,
        project=project
    )